# 01. LangChain 핵심 개념 학습

RAG 시스템 구현을 위한 LangChain 기본 개념을 단계별로 학습합니다.

## 학습 순서
1. **Document Loaders** - 문서를 LangChain 형식으로 로드
2. **Text Splitters** - 문서를 적절한 청크로 분할
3. **Embeddings** - 텍스트를 벡터로 변환
4. **Vector Stores** - 벡터 저장 및 검색
5. **Retrievers** - 관련 문서 검색
6. **Chains** - LLM + Retriever 연결

## 환경 설정

In [ ]:
import os
from dotenv import load_dotenv

# .env 파일에서 환경 변수 로드
load_dotenv()

# API 키 확인
print(f"OpenAI API Key 설정됨: {'OPENAI_API_KEY' in os.environ}")

---
## 1. Document Loaders (문서 로더)

LangChain은 다양한 소스에서 문서를 로드할 수 있습니다.
- 텍스트 파일, PDF, HTML
- 노션, Confluence, Google Drive
- 웹 페이지, GitHub 등

### Document 객체 구조
```python
Document(
    page_content="문서의 실제 텍스트 내용",
    metadata={"source": "파일 경로", "title": "제목", ...}
)
```

In [ ]:
from langchain_core.documents import Document

# Document 객체 직접 생성하기
sample_doc = Document(
    page_content="LangChain은 LLM 애플리케이션 개발을 위한 프레임워크입니다.",
    metadata={
        "source": "sample",
        "title": "LangChain 소개",
        "language": "ko"
    }
)

print(f"내용: {sample_doc.page_content}")
print(f"메타데이터: {sample_doc.metadata}")

In [ ]:
from langchain_community.document_loaders import TextLoader

# 샘플 텍스트 파일 생성
sample_text = """# Spring Boot 가이드

## 1. 프로젝트 설정
Spring Initializr를 사용하여 프로젝트를 생성합니다.
필요한 의존성: Spring Web, Spring Data JPA, H2 Database

## 2. 엔티티 생성
@Entity 어노테이션을 사용하여 JPA 엔티티를 정의합니다.

## 3. Repository
JpaRepository를 상속받아 데이터 접근 계층을 구현합니다.
"""

# 샘플 파일 저장
os.makedirs("../data/raw", exist_ok=True)
with open("../data/raw/sample_spring.txt", "w", encoding="utf-8") as f:
    f.write(sample_text)

# TextLoader로 파일 로드
loader = TextLoader("../data/raw/sample_spring.txt", encoding="utf-8")
docs = loader.load()

print(f"로드된 문서 수: {len(docs)}")
print(f"문서 내용 미리보기: {docs[0].page_content[:100]}...")

---
## 2. Text Splitters (텍스트 분할기)

긴 문서를 작은 청크로 분할합니다. 이유:
- LLM 컨텍스트 윈도우 제한
- 더 정확한 검색을 위해
- 임베딩 품질 향상

### 주요 파라미터
- `chunk_size`: 청크당 최대 문자/토큰 수
- `chunk_overlap`: 청크 간 중복되는 문자 수 (문맥 유지)

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# RecursiveCharacterTextSplitter: 가장 일반적으로 사용
# 구분자 우선순위: \n\n → \n → 공백 → 문자
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,      # 청크당 최대 200자
    chunk_overlap=50,    # 50자 중복 (문맥 유지)
    length_function=len,
    separators=["\n\n", "\n", " ", ""]  # 분할 구분자 우선순위
)

# 문서 분할
splits = text_splitter.split_documents(docs)

print(f"원본 문서 수: {len(docs)}")
print(f"분할 후 청크 수: {len(splits)}")
print("\n--- 각 청크 ---")
for i, chunk in enumerate(splits):
    print(f"\n[청크 {i+1}] (길이: {len(chunk.page_content)}자)")
    print(chunk.page_content)

In [ ]:
# 토큰 기반 분할 (OpenAI 모델 사용 시 권장)
from langchain.text_splitter import TokenTextSplitter

token_splitter = TokenTextSplitter(
    chunk_size=100,      # 청크당 최대 100 토큰
    chunk_overlap=20     # 20 토큰 중복
)

token_splits = token_splitter.split_documents(docs)
print(f"토큰 기반 분할 청크 수: {len(token_splits)}")

---
## 3. Embeddings (임베딩)

텍스트를 벡터(숫자 배열)로 변환합니다.
- 의미가 비슷한 텍스트 → 비슷한 벡터
- 벡터 간 유사도 계산으로 관련 문서 검색

### 임베딩 모델 선택
- OpenAI: `text-embedding-3-small` (1536차원, 비용 효율적)
- OpenAI: `text-embedding-3-large` (3072차원, 고품질)
- 무료: HuggingFace sentence-transformers

In [ ]:
from langchain_openai import OpenAIEmbeddings

# OpenAI 임베딩 모델 초기화
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"  # 비용 효율적인 모델
)

# 단일 텍스트 임베딩
text = "Spring Boot는 Java 기반 웹 프레임워크입니다."
vector = embeddings.embed_query(text)

print(f"텍스트: {text}")
print(f"벡터 차원: {len(vector)}")
print(f"벡터 일부: {vector[:5]}...")

In [ ]:
# 여러 텍스트 임베딩 (문서용)
texts = [
    "Spring Boot 프로젝트 설정",
    "JPA 엔티티 생성 방법",
    "Python Flask 웹 개발"  # 다른 주제
]

vectors = embeddings.embed_documents(texts)
print(f"임베딩된 문서 수: {len(vectors)}")

In [ ]:
import numpy as np

# 코사인 유사도 계산
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# 질문과 각 문서의 유사도 계산
query = "Spring Boot에서 엔티티를 어떻게 만드나요?"
query_vector = embeddings.embed_query(query)

print(f"질문: {query}\n")
for i, (text, vec) in enumerate(zip(texts, vectors)):
    similarity = cosine_similarity(query_vector, vec)
    print(f"문서 {i+1}: {text}")
    print(f"  유사도: {similarity:.4f}\n")

---
## 4. Vector Stores (벡터 저장소)

임베딩된 벡터를 저장하고 유사도 검색을 수행합니다.

### ChromaDB 선택 이유
- 로컬 파일 기반 (설치 간편)
- 소규모 데이터에 적합 (50페이지 이하)
- Python 네이티브 지원

In [ ]:
from langchain_community.vectorstores import Chroma

# 벡터 스토어 생성 (문서 + 임베딩)
vectorstore = Chroma.from_documents(
    documents=splits,           # 분할된 문서
    embedding=embeddings,       # 임베딩 모델
    persist_directory="../data/chroma"  # 저장 경로
)

print(f"벡터 스토어에 저장된 문서 수: {vectorstore._collection.count()}")

In [ ]:
# 유사도 검색
query = "JPA 엔티티를 어떻게 정의하나요?"
results = vectorstore.similarity_search(query, k=2)  # 상위 2개

print(f"질문: {query}\n")
print("--- 검색 결과 ---")
for i, doc in enumerate(results):
    print(f"\n[결과 {i+1}]")
    print(doc.page_content)

In [ ]:
# 유사도 점수와 함께 검색
results_with_scores = vectorstore.similarity_search_with_score(query, k=3)

print(f"질문: {query}\n")
for doc, score in results_with_scores:
    print(f"[점수: {score:.4f}] {doc.page_content[:50]}...")

---
## 5. Retrievers (리트리버)

Vector Store를 래핑하여 검색 인터페이스를 제공합니다.
- Chain에 연결하기 위한 표준 인터페이스
- 다양한 검색 전략 지원

In [ ]:
# VectorStore를 Retriever로 변환
retriever = vectorstore.as_retriever(
    search_type="similarity",  # 유사도 검색
    search_kwargs={"k": 3}     # 상위 3개 반환
)

# Retriever 사용
docs = retriever.invoke("Spring Boot 프로젝트를 어떻게 시작하나요?")

print(f"검색된 문서 수: {len(docs)}")
for i, doc in enumerate(docs):
    print(f"\n[문서 {i+1}] {doc.page_content[:80]}...")

---
## 6. Chains (체인) - RAG 파이프라인

Retriever + LLM을 연결하여 RAG 시스템을 구성합니다.

### RAG 흐름
```
질문 → Retriever(관련 문서 검색) → 프롬프트 구성 → LLM → 답변
```

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# LLM 초기화
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0  # 일관된 응답을 위해
)

# RAG 프롬프트 템플릿
template = """다음 컨텍스트를 기반으로 질문에 답변하세요.
컨텍스트에 없는 내용은 "문서에서 해당 정보를 찾을 수 없습니다."라고 답변하세요.

컨텍스트:
{context}

질문: {question}

답변:"""

prompt = ChatPromptTemplate.from_template(template)

print("프롬프트 템플릿이 준비되었습니다.")

In [ ]:
# 문서를 문자열로 포맷팅하는 함수
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# RAG 체인 구성 (LCEL 사용)
rag_chain = (
    {
        "context": retriever | format_docs,  # 검색 → 포맷팅
        "question": RunnablePassthrough()     # 질문 그대로 전달
    }
    | prompt    # 프롬프트 구성
    | llm       # LLM 호출
    | StrOutputParser()  # 문자열 파싱
)

print("RAG 체인이 구성되었습니다.")

In [ ]:
# RAG 체인 테스트
question = "Spring Boot에서 JPA 엔티티를 어떻게 만드나요?"
response = rag_chain.invoke(question)

print(f"질문: {question}")
print(f"\n답변: {response}")

In [ ]:
# 문서에 없는 내용 질문
question = "Django에서 모델을 어떻게 만드나요?"
response = rag_chain.invoke(question)

print(f"질문: {question}")
print(f"\n답변: {response}")

---
## 정리: RAG 전체 흐름

```
1. 문서 로드 (Document Loaders)
      ↓
2. 청크 분할 (Text Splitters)
      ↓
3. 임베딩 생성 (Embeddings)
      ↓
4. 벡터 DB 저장 (Vector Store)
      ↓
5. [사용자 질문]
      ↓
6. 유사 문서 검색 (Retriever)
      ↓
7. 프롬프트 + 문서 + 질문 조합
      ↓
8. LLM 답변 생성
```

다음 노트북에서는 노션 API를 연동하여 실제 문서를 로드하는 방법을 학습합니다.

In [ ]:
# 벡터 스토어 정리 (선택사항)
# vectorstore.delete_collection()